# 06 — Taux de fraude RÉCENT de l'émetteur

Le notebook 05 a montré que tout le signal est dans `te_origin_account` (taux de fraude
historique de l'émetteur). Ici on ajoute la version **récente** (fenêtres glissantes 5/10/20
périodes) pour coller à la dérive temporelle, et on mesure le gain honnête.

Référence : 05 (TE émetteur seul) -> last fold 0.3571, LB réel 0.3399.
Cible : last fold ~0.367+ (=> LB > 0.35).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6
TE_COLS = [C.ORIGIN_ACCT]          # seul le TE émetteur porte du signal (cf. 05)
WINDOWS = (5, 10, 20)

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    return pd.concat([X, beh, rec], axis=1)

def recent_rate(df, ref):
    return recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)

def make_model():
    try:
        from catboost import CatBoostClassifier
        return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                                  learning_rate=0.05, iterations=600, random_seed=42, verbose=False)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=42)

def add_te_train(X, ref, te_cols):
    X = X.copy()
    for col in te_cols:
        X[f"te_{col}"] = oof_target_encode_train(ref, col, C.TARGET)
    return X

def add_te_apply(X, df, ref, te_cols):
    X = X.copy()
    for col in te_cols:
        mp, gm = fit_target_map(ref, col, C.TARGET)
        X[f"te_{col}"] = apply_target_map(df, col, mp, gm)
    return X

## A/B : TE émetteur seul vs TE émetteur + taux récent

In [ ]:
def run_cv(use_recent):
    oof = np.zeros(len(train)); per_fold = []; lm = lc = None
    for tr_idx, va_idx in folds_full:
        tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
        ref = train.iloc[tr_op]
        Xtr = add_te_train(base_build(train.iloc[tr_op], ref), ref, TE_COLS)
        Xva = add_te_apply(base_build(train.iloc[va_op], ref), train.iloc[va_op], ref, TE_COLS)
        if use_recent:
            Xtr = pd.concat([Xtr, recent_rate(train.iloc[tr_op], ref)], axis=1)
            Xva = pd.concat([Xva, recent_rate(train.iloc[va_op], ref)], axis=1)
        m = make_model(); m.fit(Xtr, y_all[tr_op])
        oof[va_op] = m.predict_proba(Xva)[:, 1]
        per_fold.append(evaluate_ap(y_all[va_op], oof[va_op])); lm, lc = m, Xtr.columns
    return per_fold, oof, lm, lc

pf_te, _, _, _ = run_cv(use_recent=False)
pf_rec, oof_rec, m_rec, c_rec = run_cv(use_recent=True)

def show(name, pf):
    print(f"{name:22s} global {np.mean(pf):.4f} | recent(2) {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f}")
show("TE émetteur", pf_te)
show("+ taux récent", pf_rec)
print("\nper-fold (+récent):", [round(x, 4) for x in pf_rec])
print("Gain last fold    :", round(pf_rec[-1] - pf_te[-1], 4))
print("LB estimé (last - 0.017) :", round(pf_rec[-1] - 0.017, 4))

In [ ]:
imp = m_rec.get_feature_importance() if hasattr(m_rec, "get_feature_importance") else m_rec.feature_importances_
print(pd.Series(imp, index=c_rec).sort_values(ascending=False).round(2).head(15))

## Soumission (uploader seulement si LB estimé > 0.35)

In [ ]:
from src.calibration import fit_isotonic, apply_isotonic
from src.utils import make_submission
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")

ref_full = train.iloc[np.where(op03)[0]]
Xf = add_te_train(base_build(ref_full, ref_full), ref_full, TE_COLS)
Xf = pd.concat([Xf, recent_rate(ref_full, ref_full)], axis=1)
final = make_model(); final.fit(Xf, y_all[op03])
iso = fit_isotonic(oof_rec[op03], y_all[op03])

te_op = op03_mask(test).to_numpy()
test_op = test.iloc[np.where(te_op)[0]]
Xte = add_te_apply(base_build(test_op, ref_full), test_op, ref_full, TE_COLS)
Xte = pd.concat([Xte, recent_rate(test_op, ref_full)], axis=1)
proba = apply_isotonic(iso, final.predict_proba(Xte)[:, 1])
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "06_recent_origin")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))